# Fraud Detection

**Oasis Infobyte Data Analytics Internship — Level 2, Task 3**

This notebook builds a leakage-safe fraud-detection pipeline using Logistic Regression and Random Forest, with SMOTE applied only to the training data.


## 1. Import Libraries


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    roc_curve
)

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)


## 2. Load Dataset


In [ ]:
DATA_PATH = "creditcard.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())


## 3. Class Imbalance Analysis


In [ ]:
class_counts = df["Class"].value_counts().sort_index()
fraud_count = int(class_counts.get(1, 0))
total_count = len(df)
fraud_percentage = (fraud_count / total_count) * 100

print("Total transactions:", total_count)
print("Fraudulent transactions:", fraud_count)
print(f"Fraud percentage: {fraud_percentage:.4f}%")

class_summary = pd.DataFrame({
    "Count": class_counts,
    "Percentage": (class_counts / total_count * 100).round(4)
})

display(class_summary)

plt.figure(figsize=(7, 5))
sns.countplot(data=df, x="Class")
plt.title("Transaction Class Distribution")
plt.xlabel("Class (0 = Non-Fraud, 1 = Fraud)")
plt.ylabel("Count")
plt.tight_layout()
plt.show()


## 4. Why Accuracy Is Misleading

In a heavily imbalanced fraud dataset, a model can classify almost every transaction as non-fraud and still obtain very high accuracy.

For example, if fraudulent transactions represent only a tiny fraction of all transactions, correctly predicting the large non-fraud class can dominate the accuracy calculation.

Fraud detection therefore requires metrics that specifically examine the minority class, particularly Precision, Recall and F1-score. ROC-AUC is also useful for evaluating the ranking ability of a probability-based classifier across thresholds.

The business objective determines the desired Precision/Recall trade-off.


## 5. Transaction Amount Analysis


In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(
    data=df,
    x="Class",
    y="Amount"
)
plt.yscale("log")
plt.title("Transaction Amount by Fraud Class")
plt.xlabel("Class (0 = Non-Fraud, 1 = Fraud)")
plt.ylabel("Transaction Amount (log scale)")
plt.tight_layout()
plt.show()

amount_summary = df.groupby("Class")["Amount"].agg(
    ["count", "mean", "median", "min", "max"]
)

display(amount_summary)


## 6. Time-of-Day Analysis


In [ ]:
# The Time column records seconds elapsed from the first transaction.
# Converting it to hours provides an approximate within-day time-of-day view.

seconds_per_day = 24 * 60 * 60

df["HourOfDay"] = ((df["Time"] % seconds_per_day) / 3600).astype(int)

hourly_fraud = (
    df.groupby("HourOfDay")["Class"]
    .agg(["count", "sum"])
    .rename(columns={"sum": "fraud_count"})
)

hourly_fraud["fraud_rate_pct"] = (
    hourly_fraud["fraud_count"] / hourly_fraud["count"] * 100
)

plt.figure(figsize=(12, 5))
sns.lineplot(
    data=hourly_fraud,
    x=hourly_fraud.index,
    y="fraud_rate_pct",
    marker="o"
)
plt.title("Fraud Rate by Approximate Hour of Day")
plt.xlabel("Hour of Day")
plt.ylabel("Fraud Rate (%)")
plt.xticks(range(24))
plt.tight_layout()
plt.show()

display(hourly_fraud)


## 7. Prepare Features and Stratified Split

The target is `Class`.

The `HourOfDay` feature is retained as an engineered time feature.

The dataset is split with stratification so that both training and test sets contain fraud cases.

**Important:** SMOTE is applied after the split and only within the training pipeline. The test set is never oversampled.


In [ ]:
X = df.drop(columns=["Class"])
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

print("\nTraining class distribution:")
display(y_train.value_counts().to_frame("count"))

print("\nTesting class distribution:")
display(y_test.value_counts().to_frame("count"))


## 8. Train Logistic Regression with SMOTE


In [ ]:
logistic_pipeline = ImbPipeline([
    ("scaler", StandardScaler()),
    ("smote", SMOTE(random_state=42)),
    ("model", LogisticRegression(
        max_iter=2000,
        random_state=42
    ))
])

logistic_pipeline.fit(X_train, y_train)

logistic_pred = logistic_pipeline.predict(X_test)
logistic_proba = logistic_pipeline.predict_proba(X_test)[:, 1]

logistic_metrics = {
    "Model": "Logistic Regression",
    "Accuracy": accuracy_score(y_test, logistic_pred),
    "Precision": precision_score(y_test, logistic_pred, zero_division=0),
    "Recall": recall_score(y_test, logistic_pred, zero_division=0),
    "F1-Score": f1_score(y_test, logistic_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, logistic_proba)
}

display(pd.DataFrame([logistic_metrics]).round(4))


## 9. Train Random Forest with SMOTE


In [ ]:
random_forest_pipeline = ImbPipeline([
    ("smote", SMOTE(random_state=42)),
    ("model", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
        class_weight=None
    ))
])

random_forest_pipeline.fit(X_train, y_train)

rf_pred = random_forest_pipeline.predict(X_test)
rf_proba = random_forest_pipeline.predict_proba(X_test)[:, 1]

rf_metrics = {
    "Model": "Random Forest",
    "Accuracy": accuracy_score(y_test, rf_pred),
    "Precision": precision_score(y_test, rf_pred, zero_division=0),
    "Recall": recall_score(y_test, rf_pred, zero_division=0),
    "F1-Score": f1_score(y_test, rf_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, rf_proba)
}

display(pd.DataFrame([rf_metrics]).round(4))


## 10. Classification Reports


In [ ]:
print("=" * 70)
print("Logistic Regression")
print("=" * 70)
print(classification_report(
    y_test,
    logistic_pred,
    target_names=["Non-Fraud", "Fraud"],
    digits=4
))

print("=" * 70)
print("Random Forest")
print("=" * 70)
print(classification_report(
    y_test,
    rf_pred,
    target_names=["Non-Fraud", "Fraud"],
    digits=4
))


## 11. Confusion Matrices


In [ ]:
for name, predictions in [
    ("Logistic Regression", logistic_pred),
    ("Random Forest", rf_pred)
]:
    cm = confusion_matrix(y_test, predictions)

    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        xticklabels=["Non-Fraud", "Fraud"],
        yticklabels=["Non-Fraud", "Fraud"]
    )
    plt.title(f"Confusion Matrix — {name}")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.show()


## 12. ROC-AUC Curves


In [ ]:
logistic_fpr, logistic_tpr, _ = roc_curve(y_test, logistic_proba)
rf_fpr, rf_tpr, _ = roc_curve(y_test, rf_proba)

plt.figure(figsize=(8, 6))
plt.plot(
    logistic_fpr,
    logistic_tpr,
    label=f"Logistic Regression (AUC = {logistic_metrics['ROC-AUC']:.4f})"
)
plt.plot(
    rf_fpr,
    rf_tpr,
    label=f"Random Forest (AUC = {rf_metrics['ROC-AUC']:.4f})"
)
plt.plot([0, 1], [0, 1], linestyle="--")

plt.title("ROC Curves")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.tight_layout()
plt.show()


## 13. Which Metric Matters Most?

There is no universal single metric for every fraud operation.

**Recall** measures the proportion of actual fraudulent transactions that are detected. High recall reduces missed fraud, or false negatives.

**Precision** measures the proportion of transactions flagged as fraud that are actually fraudulent. High precision reduces false alerts.

If missed fraud is substantially more costly than customer friction, a system may prioritise recall. If false positives create major customer or operational costs, precision becomes more important.

A practical fraud system often selects a probability threshold based on the monetary cost of false negatives and false positives rather than simply using the default 0.50 threshold.


## 14. Feature Importance — Random Forest


In [ ]:
rf_model = random_forest_pipeline.named_steps["model"]

feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf_model.feature_importances_
}).sort_values("Importance", ascending=False)

display(feature_importance.head(20))

plt.figure(figsize=(10, 8))
sns.barplot(
    data=feature_importance.head(15),
    x="Importance",
    y="Feature"
)
plt.title("Top Random Forest Feature Importances")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()


## 15. Coefficient Analysis — Logistic Regression


In [ ]:
scaler = logistic_pipeline.named_steps["scaler"]
logistic_model = logistic_pipeline.named_steps["model"]

coefficient_df = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": logistic_model.coef_[0]
})

coefficient_df["AbsoluteCoefficient"] = coefficient_df["Coefficient"].abs()

print("Largest positive coefficients:")
display(
    coefficient_df
    .sort_values("Coefficient", ascending=False)
    .head(10)
)

print("Largest negative coefficients:")
display(
    coefficient_df
    .sort_values("Coefficient", ascending=True)
    .head(10)
)


## 16. Model Comparison


In [ ]:
comparison = pd.DataFrame([
    logistic_metrics,
    rf_metrics
])

display(comparison.round(4))

comparison.to_csv(
    "fraud_model_comparison.csv",
    index=False
)

feature_importance.to_csv(
    "random_forest_feature_importance.csv",
    index=False
)

coefficient_df.to_csv(
    "logistic_regression_coefficients.csv",
    index=False
)


## 17. Scalability — 1 Million Transactions per Hour

One million transactions per hour is approximately 278 transactions per second.

A production fraud-detection system at that scale would need to separate offline model training from online inference.

A scalable architecture could include:

1. Streaming ingestion for transaction events.
2. Fast feature computation using precomputed or low-latency features.
3. A lightweight model-serving layer for real-time predictions.
4. Horizontal scaling across multiple inference workers.
5. Threshold and rules engines for prioritising alerts.
6. Monitoring for latency, drift, fraud-rate changes and model degradation.
7. Periodic retraining using newly labelled fraud outcomes.
8. A feedback loop from confirmed fraud investigations.

The notebook demonstrates the modelling workflow but is not itself a production-grade real-time serving system.


## 18. Final Conclusion


In [ ]:
print("Fraud detection modelling completed.")
print("Primary evaluation metrics: Precision, Recall, F1-Score and ROC-AUC.")
print("Review the comparison table and confusion matrices before selecting a production threshold.")
